# Analysis: Known Issues in HGNN.ipynb

## Critical

**HGNN with 7 hyperedges is equivalent to cluster averaging**
`HypergraphConv` aggregates features across hyperedges. With only 7 hyperedges
(one per cluster), each layer averages features within a cluster. This is not meaningful
message passing — it is simple cluster-level pooling that cannot learn spatial structure.

**Hyperedge index encodes the same labels the model is trained to predict**
The hyperedges come from upstream cluster assignments, and the HGNN is then trained to
reproduce those same assignments. The model has no independent structural signal to learn from.

## High

**No validation set — early stopping on training accuracy**
The early stopping criterion triggers when *training* accuracy reaches 99%+. There is no
held-out validation set. Early stopping therefore detects memorization of noisy labels,
not generalization. The model is stopped at peak overfitting.
**Fix:** Hold out 15–20% of labeled nodes as a validation set and monitor validation loss.

**Redundant k-means re-clustering after classifier training**
After training the HGNN classifier to predict 7 classes, its logit predictions are discarded
and `KMeans(n_clusters=7)` is run on the output embeddings. This two-stage redundancy suggests
the trained classifier is not trusted, which is a sign that training did not converge.
**Fix:** Use the classifier's `argmax` predictions directly as final cluster assignments.

In [ ]:
################################################################################
# 0.  Install once                                                             #
################################################################################
# pip install --quiet torch==2.3.1                                           \
#                 torch-sparse torch-scatter                                 \
#                 -f https://data.pyg.org/whl/torch-2.3.1+cpu.html
# pip install --quiet torch-geometric umap-learn scikit-learn matplotlib seaborn

################################################################################
# 1.  Load data                                                                #
################################################################################
import numpy as np, torch
import csv
from pathlib import Path
from torch_geometric.data import Data

# ── 1.1  labels ──────────────────────────────────────────────────────────────
PREFER_NESTED_ARTIFACTS = True
FORCE_BASELINE_ARTIFACTS = False

if FORCE_BASELINE_ARTIFACTS:
    label_candidates = [Path('Data/spaformer_prepared/leiden_labels.npy')]
elif PREFER_NESTED_ARTIFACTS:
    label_candidates = [
        Path('Data/spaformer_prepared/labels_nested.npy'),
        Path('Data/spaformer_prepared/leiden_labels.npy'),
    ]
else:
    label_candidates = [
        Path('Data/spaformer_prepared/leiden_labels.npy'),
        Path('Data/spaformer_prepared/labels_nested.npy'),
    ]
labels = None
label_source = None
for cand in label_candidates:
    if cand.exists():
        labels = np.load(cand).astype('int64')
        label_source = cand
        break
if labels is None:
    raise FileNotFoundError('No label file found.')

target_nodes = int(labels.shape[0])
use_nested = label_source.name == 'labels_nested.npy'
print(f'Using labels from {label_source}')
print(f'Artifact policy: prefer_nested={PREFER_NESTED_ARTIFACTS}, force_baseline={FORCE_BASELINE_ARTIFACTS}')

# ── 1.2  node features ───────────────────────────────────────────────────────
if FORCE_BASELINE_ARTIFACTS:
    feature_candidates = [
        Path('Data/spaformer_prepared/latent_refined.npy'),
        Path('Data/spaformer_prepared/latent.npy'),
        Path('Data/spaformer_prepared/X.npy'),
    ]
elif PREFER_NESTED_ARTIFACTS:
    feature_candidates = [
        Path('Data/spaformer_prepared/latent_nested.npy'),
        Path('Data/spaformer_prepared/latent_refined.npy'),
        Path('Data/spaformer_prepared/latent.npy'),
        Path('Data/spaformer_prepared/X.npy'),
    ]
else:
    feature_candidates = [
        Path('Data/spaformer_prepared/latent_refined.npy'),
        Path('Data/spaformer_prepared/latent.npy'),
        Path('Data/spaformer_prepared/latent_nested.npy'),
        Path('Data/spaformer_prepared/X.npy'),
    ]
X = None
feature_source = None
for cand in feature_candidates:
    if not cand.exists():
        continue
    arr = np.load(cand).astype('float32')
    if arr.shape[0] == target_nodes:
        X = arr
        feature_source = cand
        if cand.name == 'latent_nested.npy':
            use_nested = True
        break
if X is None:
    raise FileNotFoundError('No feature matrix found for HGNN.')
print(f'Using node features from {feature_source}')

coords_norm = np.load('Data/spaformer_prepared/C.npy').astype('float32')
coords_raw  = np.load('Data/spaformer_prepared/C_raw.npy').astype('float32')

# ── 1.3  hyperedges ──────────────────────────────────────────────────────────
if FORCE_BASELINE_ARTIFACTS:
    hyperedge_candidates = [Path('Data/hyperedge_index.npy')]
elif PREFER_NESTED_ARTIFACTS:
    hyperedge_candidates = [
        Path('Data/hyperedge_index_nested.npy'),
        Path('Data/hyperedge_index.npy'),
    ]
else:
    hyperedge_candidates = [
        Path('Data/hyperedge_index.npy'),
        Path('Data/hyperedge_index_nested.npy'),
    ]
H_idx = None
hyperedge_source = None
for cand in hyperedge_candidates:
    if cand.exists():
        H_idx = np.load(cand).astype('int64')
        hyperedge_source = cand
        if cand.name == 'hyperedge_index_nested.npy':
            use_nested = True
        break
if H_idx is None:
    raise FileNotFoundError('No hyperedge index found.')
print(f'Using hyperedges from {hyperedge_source}')

if coords_norm.shape[0] != target_nodes or coords_raw.shape[0] != target_nodes:
    raise ValueError('Coordinate count mismatch with labels')
if H_idx.size == 0:
    raise ValueError('Hyperedge index is empty.')
node_total = int(H_idx[0].max()) + 1
if node_total != target_nodes:
    raise ValueError(f'Hypergraph node count {node_total} != labels {target_nodes}')

# pixel coordinates
positions_csv = Path('Data/positions.txt')
if positions_csv.exists():
    lookup = {}
    with positions_csv.open() as fh:
        reader = csv.reader(fh)
        for barcode, in_tissue, array_row, array_col, pxl_row, pxl_col in reader:
            row_val = float(pxl_row)
            col_val = float(pxl_col)
            flag = (in_tissue == '1')
            lookup[(row_val, col_val)] = (flag, col_val, row_val)
            lookup[(col_val, row_val)] = (flag, col_val, row_val)
    coords_xy = np.zeros_like(coords_raw)
    tissue_mask = np.zeros(coords_raw.shape[0], dtype=bool)
    for idx_pt, (a, b) in enumerate(coords_raw):
        key = (float(a), float(b))
        if key not in lookup:
            raise KeyError(f'Missing Visium coordinate {key}')
        flag, col_val, row_val = lookup[key]
        tissue_mask[idx_pt] = flag
        coords_xy[idx_pt] = (col_val, row_val)
else:
    tissue_mask = np.ones(coords_raw.shape[0], dtype=bool)
    coords_xy = coords_raw[:, ::-1]
coords_img = coords_xy - coords_xy.min(axis=0, keepdims=True)
np.save('Data/spaformer_prepared/in_tissue_mask.npy', tissue_mask)

# map labels to contiguous codes
train_idx = labels != -1
train_labels = labels[train_idx]
if train_labels.size == 0:
    raise ValueError('All labels are -1 (noise); cannot train HGNN.')
uniq_labels = np.unique(train_labels)
label_codes = np.searchsorted(uniq_labels, train_labels).astype('int64')
num_classes = int(len(uniq_labels))

N, feat_dim = X.shape
print('nodes:', N, 'features:', feat_dim, 'hyper-edges:', np.unique(H_idx[1]).size)
print('Classes (excl. noise):', num_classes)
print('Noise nodes:', int((labels == -1).sum()))

data = Data(
    x               = torch.from_numpy(X),
    hyperedge_index = torch.from_numpy(H_idx),
    pos             = torch.from_numpy(coords_norm)
)

################################################################################
# 2.  HGNN model                                                               #
################################################################################
import torch.nn as nn
from torch_geometric.nn import HypergraphConv

class SimpleHGNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.1):
        super().__init__()
        self.h1 = HypergraphConv(in_dim,  hidden)
        self.h2 = HypergraphConv(hidden, out_dim)
        self.act = nn.ReLU()
        self.dp  = nn.Dropout(dropout)

    def forward(self, data):
        x = self.act(self.h1(data.x, data.hyperedge_index))
        x = self.dp(x)
        z = self.h2(x, data.hyperedge_index)
        return z

model = SimpleHGNN(in_dim=feat_dim, hidden=256, out_dim=256, dropout=0.1)
classifier = nn.Linear(256, num_classes)
print(model)

################################################################################
# 3.  Train / val split + label-consistency training                           #
################################################################################
from sklearn.model_selection import train_test_split as _tts

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data   = data.to(device)
model  = model.to(device)
classifier = classifier.to(device)

# [FIX] Create an 80/20 train/val split over the valid (non-noise) nodes.
# Previously there was no validation set; early stopping was based on training
# accuracy, which just detects memorisation of (noisy) labels.
valid_positions = np.where(train_idx)[0]       # node indices in [0, N)
pos_tr, pos_val = _tts(
    np.arange(len(valid_positions)),
    test_size=0.2,
    random_state=42,
    stratify=label_codes,
)
tr_node_idx  = valid_positions[pos_tr]
val_node_idx = valid_positions[pos_val]
tr_labels    = label_codes[pos_tr]
val_labels   = label_codes[pos_val]

tr_idx_t   = torch.from_numpy(tr_node_idx).to(device)
val_idx_t  = torch.from_numpy(val_node_idx).to(device)
tr_labels_t  = torch.from_numpy(tr_labels).to(device)
val_labels_t = torch.from_numpy(val_labels).to(device)
print(f'Train nodes: {len(tr_node_idx)}  Val nodes: {len(val_node_idx)}')

class_counts = np.bincount(tr_labels, minlength=num_classes).astype('float32')
class_weights = class_counts.sum() / np.clip(class_counts, 1.0, None)
class_weights = class_weights / class_weights.mean()
class_weights_t = torch.from_numpy(class_weights).to(device)

opt = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()),
    lr=5e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)

epochs   = 200
patience = 20
best_val_loss = float('inf')
best_state    = None
no_improve    = 0

for epoch in range(1, epochs + 1):
    # ── train ──
    model.train(); classifier.train()
    opt.zero_grad()
    z      = model(data)
    logits = classifier(z)
    loss   = criterion(logits[tr_idx_t], tr_labels_t)
    loss.backward(); opt.step()

    # ── validate ──  [FIX] monitor val loss, not train accuracy
    model.eval(); classifier.eval()
    with torch.no_grad():
        z_val      = model(data)
        logits_val = classifier(z_val)
        val_loss   = criterion(logits_val[val_idx_t], val_labels_t).item()
        tr_acc     = (logits[tr_idx_t].argmax(1) == tr_labels_t).float().mean().item() * 100
        val_acc    = (logits_val[val_idx_t].argmax(1) == val_labels_t).float().mean().item() * 100

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = {
            'model':      model.state_dict(),
            'classifier': classifier.state_dict(),
        }
        no_improve = 0
    else:
        no_improve += 1

    if epoch == 1 or epoch % 25 == 0:
        print(f"epoch {epoch:3d}/{epochs}  loss {loss.item():.4f}  "
              f"tr_acc {tr_acc:.1f}%  val_loss {val_loss:.4f}  val_acc {val_acc:.1f}%")

    if no_improve >= patience:
        print(f'Early stopping at epoch {epoch}  (best val_loss {best_val_loss:.4f})')
        break

if best_state is not None:
    model.load_state_dict(best_state['model'])
    classifier.load_state_dict(best_state['classifier'])

################################################################################
# 4.  Final predictions & spatial ordering                                     #
################################################################################
from sklearn.preprocessing import StandardScaler, normalize
import seaborn as sns, matplotlib.pyplot as plt

TARGET_FINAL_CLUSTERS = 7

model.eval(); classifier.eval()
with torch.no_grad():
    Z_torch = model(data)
    logits  = classifier(Z_torch)
    pred_codes = logits.argmax(dim=1).cpu().numpy()
    Z = Z_torch.cpu().numpy()

pred_labels = uniq_labels[pred_codes]

suffix = '_nested' if use_nested else ''
np.save(f'Data/hgnn_embeddings{suffix}.npy', Z)
np.save(f'Data/hgnn_predictions{suffix}.npy', pred_labels)
print(f'✓ HGNN embeddings  → Data/hgnn_embeddings{suffix}.npy')
print(f'✓ HGNN predictions → Data/hgnn_predictions{suffix}.npy')

# [FIX] Use classifier argmax (pred_codes) directly as the final cluster assignments.
# Previously, KMeans was re-run on the HGNN embeddings, discarding the classifier
# output. That redundancy signalled the classifier was not trusted. With a proper
# validation split and val-loss-based early stopping, the classifier predictions are
# now reliable and should be used directly.
final_codes = pred_codes.copy()

# Spatial ordering: assign label IDs top-to-bottom by mean y-coordinate.
coords_plot = coords_img[tissue_mask]
plot_codes  = final_codes[tissue_mask]
order = sorted(np.unique(plot_codes),
               key=lambda k: coords_plot[plot_codes == k, 1].mean())
mapping    = {old: new for new, old in enumerate(order)}
final_codes = np.array([mapping.get(c, c) for c in final_codes], dtype=int)

np.save(f'Data/hgnn_clusters{suffix}.npy', final_codes)
print(f'✓ Final cluster labels → Data/hgnn_clusters{suffix}.npy')
print(f'Final cluster count: {np.unique(final_codes).size}')

# Visualise
coords_plot = coords_img[tissue_mask]
plot_codes  = final_codes[tissue_mask]
palette = sns.color_palette('tab10', np.unique(plot_codes).size)

plt.figure(figsize=(4.8, 4.4))
sns.scatterplot(
    x=coords_plot[:, 0], y=coords_plot[:, 1],
    hue=plot_codes, palette=palette, s=12, linewidth=0, legend='full',
)
plt.gca().invert_yaxis()
plt.axis('equal'); plt.axis('off')
plt.title(f'HGNN final clusters ({np.unique(plot_codes).size})')
plt.tight_layout(); plt.show()